In [1]:
import numpy
print(numpy.__version__)
print(numpy.__file__)

import sys
print(sys.executable)

2.5.0
/home/nehadesigar/.local/lib/python3.12/site-packages/numpy/__init__.py
/home/nehadesigar/pixi_env/.pixi/envs/default/bin/python


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
# from numba import jit, prange

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
from scipy.signal import find_peaks
from scipy.integrate import quad
from scipy.optimize import root_scalar
from matplotlib.animation import PillowWriter, FuncAnimation
import time

In [3]:
# Independent parameters (free to edit)

Na = 0.5 # Units: M 
T = 303.15 # Units: K
valence = 4
duration = 500 * 10**5 # In timesteps of dt
gridpoints = 256 # Number of points
dx = 10 # Units: nm
dt = 1.E-5 # Units: sec
rho_mean = 9E-5 # Initial mean density of nanostar A, found by spinodal (rho dense + rho dilute)/2 for the value of T used
save_interval = 10**5

grid_length = dx * gridpoints # Total length (nm)
inv_dx2= 1.0 / (dx * dx)

#Establishes constants
M = 1 # Units: (nm s)^-1
vb = 1.66 # Units: nm^3
kB = 1.314E-23*0.24 # Units: cal/K (1J=0.24cal)
mol = 6.02E23
floor = 1E-12 # Minimum value for arrays
num_saves = duration // save_interval + 1 #Number of saved values

# Setup for AAAA vs ABBB System
B2 = 1600 # Units: nm^3
dHa = -40000 # Units: cal/mol 
dSa = -118 # Units: cal/mol K
dHb = -42000 # Units: cal/mol 
dSb = -120 # Units: cal/mol K

K = 1.0E6 # Units: nm^5 

type = "AAAA"
temperatures = np.arange(290.15,310.15, 0.5)

In [4]:
n_temperatures = len(temperatures)  # temperatures = array of T values to sweep over

# Creates rho values around the mean with slight randomness, one row per temperature
rho_all = np.zeros((n_temperatures, gridpoints))
for i in range(n_temperatures):
    rho_all[i] = rho_mean * (np.concatenate([np.ones(gridpoints // 2), np.zeros(gridpoints - gridpoints // 2)]))
rho_all = np.maximum(rho_all, 1.E-10)  # Prevents negative densities
initial_mass = np.sum(rho_all, axis=1)

# Varies with temperature
Da_all = vb * np.exp(-(dHa - temperatures * (dSa)) / (mol * kB * temperatures))
Db_all = vb * np.exp(-(dHb - temperatures * (dSb)) / (mol * kB * temperatures))

# @jit(nopython=True, parallel = False, cache = False)
def laplacian_1d(function_array):
    """
    Computes the 1D Laplacian of a function, given an array representing that function
    """
    return (np.roll(function_array, -1) - 2*function_array + np.roll(function_array, 1)) * inv_dx2
    # Note: I used 'roll' so it would have periodic boundary conditions


# @jit(nopython=True, parallel = False, cache = False) # Converts the given function into machine code (optimization)
def compute_step_single_AAAA(rho, Da):
    # Total chemical potential (with floored rho, Xa)
    beta_mu_total = (2.0 * B2 * rho + np.log(rho) + #beta mu_ref
                    valence * np.log((-1 + np.sqrt(1 + 4 * 4*rho*Da)) / (2 * 4*rho*Da)) - #beta mu_b
                    K * (np.roll(rho, -1) - 2.0 * rho + np.roll(rho, 1)) * inv_dx2) #beta mu_int
    # Finds the 1D Laplacian of beta mu total
    laplacian_1d_mu = laplacian_1d(beta_mu_total)
    # Updates the density explicitly: rho(t+dt) = rho(t) + dt * M laplacian (beta mu_total)
    return  dt * M * laplacian_1d_mu


# @jit(nopython=True, parallel = False, cache = False) # Converts the given function into machine code (optimization)
def compute_step_single_ABBB(rho, Da, Db):
    # Arm-type bonding concentrations: 1 A-arm and 3 B-arms per nanostar, no AAAA species present
    Ca = rho * Da          
    Cb = 3.0 * rho * Db    
    Xa = (-1.0 + np.sqrt(1.0 + 4.0 * Ca)) / (2.0 * Ca)  # unbonded fraction, A arms
    Xb = (-1.0 + np.sqrt(1.0 + 4.0 * Cb)) / (2.0 * Cb)  # unbonded fraction, B arms

    # Total chemical potential (with floored rho, Xa, Xb)
    beta_mu_total = (2.0 * B2 * rho + np.log(rho) +               #beta mu_ref
                    np.log(Xa) + 3.0 * np.log(Xb) -                #beta mu_b (1 A-arm + 3 B-arms)
                    K * (np.roll(rho, -1) - 2.0 * rho + np.roll(rho, 1)) * inv_dx2) #beta mu_int
    # Finds the 1D Laplacian of beta mu total
    laplacian_1d_mu = laplacian_1d(beta_mu_total)
    # Updates the density explicitly: rho(t+dt) = rho(t) + dt * M laplacian (beta mu_total)
    return  dt * M * laplacian_1d_mu



# @jit(nopython=True, parallel = True, cache = False)
def apply_timestep_all_AAAA(rho_all, Da_all):
    # Applies one timestep to every temperature row in parallel
    n_temperatures, gridpoints = rho_all.shape
    rho_out = np.zeros_like(rho_all)
    for i in range(n_temperatures):              #prange
        rho = rho_all[i]
        Da = Da_all[i]
        rho_out[i] = rho + compute_step_single_AAAA(rho, Da)
    return rho_out

# @jit(nopython=True, parallel = True, cache = False)
def apply_timestep_all_ABBB(rho_all, Da_all, Db_all):
    # Applies one timestep to every temperature row in parallel
    n_temperatures, gridpoints = rho_all.shape
    rho_out = np.zeros_like(rho_all)
    for i in range(n_temperatures):              #prange
        rho = rho_all[i]
        Da = Da_all[i]
        Db = Db_all[i]
        rho_out[i] = rho + compute_step_single_ABBB(rho, Da, Db)
    return rho_out

def free_energy_AAAA(rho_arr, Da_local, B2, K, dx, floor, valence):
    rho_f = np.maximum(rho_arr, floor)
    Ca = np.maximum(4 * rho_f * Da_local, floor)
    Xa = (-1 + np.sqrt(1 + 4 * Ca)) / (2 * Ca)
    f_ref = rho_f * np.log(rho_f) - rho_f + B2 * rho_f**2
    f_b = rho_f * valence * (np.log(Xa) + (1 - Xa) / 2)
    f_grad = 0.5 * K * (np.gradient(rho_f, dx))**2
    return np.trapezoid(f_ref + f_b + f_grad, dx=dx)

def free_energy_ABBB(rho_arr, Da_local, Db_local, B2, K, dx, floor):
    rho_f = np.maximum(rho_arr, floor)
    Ca = np.maximum(rho_f * Da_local, floor)
    Cb = np.maximum(3.0 * rho_f * Db_local, floor)
    Xa = (-1 + np.sqrt(1 + 4 * Ca)) / (2 * Ca)
    Xb = (-1 + np.sqrt(1 + 4 * Cb)) / (2 * Cb)
    f_ref = rho_f * np.log(rho_f) - rho_f + B2 * rho_f**2
    f_b = rho_f * (np.log(Xa) + 3.0 * np.log(Xb))
    f_grad = 0.5 * K * (np.gradient(rho_f, dx))**2
    return np.trapezoid(f_ref + f_b + f_grad, dx=dx)

def compute_surface_tension(rho, Da, dx, floor, B2, K, valence, model_type, Db=None):
    rho_f = np.maximum(rho, floor)
    A = dx**2

    rho_dilute = np.min(rho_f)
    rho_dense  = np.max(rho_f)
    rho_dilute_sys = np.full_like(rho_f, rho_dilute)
    rho_dense_sys  = np.full_like(rho_f, rho_dense)

    if model_type == "AAAA":
        F_dilute = free_energy_AAAA(rho_dilute_sys, Da, B2, K, dx, floor, valence)
        F_dense = free_energy_AAAA(rho_dense_sys, Da, B2, K, dx, floor, valence)
        F_coex = free_energy_AAAA(rho_f, Da, B2, K, dx, floor, valence)
    else:
        F_dilute = free_energy_ABBB(rho_dilute_sys, Da, Db, B2, K, dx, floor)
        F_dense = free_energy_ABBB(rho_dense_sys, Da, Db, B2, K, dx, floor)
        F_coex = free_energy_ABBB(rho_f, Da, Db, B2, K, dx, floor)

    threshold = (rho_dilute + rho_dense) / 2
    is_dense = rho_f >= threshold
    dilute_fraction = (len(rho_f) - np.sum(is_dense)) / len(rho_f)
    dense_fraction  = np.sum(is_dense)  / len(rho_f)

    # periodic interface count: wrap last -> first
    is_dense_wrapped = np.concatenate([is_dense, is_dense[:1]])
    interface_count = np.sum(np.diff(is_dense_wrapped.astype(int)) != 0)

    if interface_count == 0:
        return 0  # homogeneous system, no interface — flag rather than divide by zero

    return (F_coex - dilute_fraction * F_dilute - dense_fraction * F_dense) / (interface_count * A)

In [ ]:
log_dir = f"OUTPUTS/1D_surface_tension_{type}"
os.makedirs(log_dir, exist_ok=True)
progress_file = os.path.join(log_dir, "1D_sequence_surface_tension.txt")

with open(progress_file, "w") as f:
            f.write(f"Parameters type = {type}, B2 = {B2}, dHa = {dHa}, dSa = {dSa}, dHb = {dHb}, dSb = {dSb}")

start_time = time.perf_counter()

if type == "AAAA":
    for step in range(duration):
    
        # Iterates to find new value of rho_all, all temperatures at once
        rho_all = apply_timestep_all_AAAA(rho_all, Da_all)
        if step % save_interval == 0:
            rho_all = np.maximum(rho_all, floor)
            with open(progress_file, "w") as f:
                f.write(f"Parameters type = {type}, B2 = {B2}, dHa = {dHa}, dSa = {dSa}, dHb = {dHb}, dSb = {dSb}")
            with open(progress_file, "a") as f:
                f.write(f"Progress: {step//save_interval} out of {duration//save_interval}\n")

            
            surface_tensions = np.full(n_temperatures, np.nan)
            
            for i in range(n_temperatures):
                surface_tensions[i] = compute_surface_tension(
                    rho_all[i], Da_all[i], dx, floor, B2, K, valence, "AAAA"
                )
                plt.clf()
                plt.plot(temperatures, surface_tensions, 'o-')
                plt.xlabel("T")
                plt.ylabel("Surface tension")
                plt.savefig(os.path.join(log_dir, f"surface_tension_vs_T.png"))
                
            x = np.arange(gridpoints) * dx
            
            # Select a subset of temperature indices, evenly spaced across the sweep
            n_plots = 15  # adjust as needed
            plot_indices = np.linspace(0, n_temperatures - 1, n_plots, dtype=int)
            plot_indices = np.unique(plot_indices)  # in case n_temperatures < n_plots
            n_plots = len(plot_indices)
            
            # Grid layout: roughly square
            n_cols = int(np.ceil(np.sqrt(n_plots)))
            n_rows = int(np.ceil(n_plots / n_cols))
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3 * n_rows), sharex=True, sharey=True)
            axes = np.atleast_1d(axes).flatten()
            
            for ax, i in zip(axes, plot_indices):
                rho_plot = np.maximum(rho_all[i], floor)
                ax.plot(x, rho_plot, lw=1)
                ax.set_title(f"T = {temperatures[i]:.4g}", fontsize=9)
                ax.set_xlabel("x")
                ax.set_ylabel(r"$\rho(x)$")
            
            # Turn off unused axes if grid has more slots than plots
            for ax in axes[n_plots:]:
                ax.axis("off")
            
            fig.suptitle(f"Density profiles ({type})")
            fig.tight_layout()
            fig.savefig(os.path.join(log_dir, "rho_profiles_subplots.png"), dpi=200)
            plt.close(fig)




            
elif type == "ABBB":
    for step in range(duration):
    
        # Iterates to find new value of rho_all, all temperatures at once
        rho_all = apply_timestep_all_ABBB(rho_all, Da_all, Db_all)
        if step % save_interval == 0:
            rho_all = np.maximum(rho_all, floor)
            with open(progress_file, "w") as f:
                f.write(f"Parameters type = {type}, B2 = {B2}, dHa = {dHa}, dSa = {dSa}, dHb = {dHb}, dSb = {dSb}")
            with open(progress_file, "a") as f:
                f.write(f"Progress: {step//save_interval} out of {duration//save_interval}\n")

            surface_tensions = np.full(n_temperatures, np.nan)
            
            for i in range(n_temperatures):
                surface_tensions[i] = compute_surface_tension(
                    rho_all[i], Da_all[i], dx, floor, B2, K, valence, "ABBB", Db=Db_all[i]
                )
                plt.clf()
                plt.plot(temperatures, surface_tensions, 'o-')
                plt.xlabel("T")
                plt.ylabel("Surface tension")
                plt.savefig(os.path.join(log_dir, f"surface_tension_vs_T.png"))

            x = np.arange(gridpoints) * dx
            
            # Select a subset of temperature indices, evenly spaced across the sweep
            n_plots = 15  # adjust as needed
            plot_indices = np.linspace(0, n_temperatures - 1, n_plots, dtype=int)
            plot_indices = np.unique(plot_indices)  # in case n_temperatures < n_plots
            n_plots = len(plot_indices)
            
            # Grid layout: roughly square
            n_cols = int(np.ceil(np.sqrt(n_plots)))
            n_rows = int(np.ceil(n_plots / n_cols))
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3 * n_rows), sharex=True, sharey=True)
            axes = np.atleast_1d(axes).flatten()
            
            for ax, i in zip(axes, plot_indices):
                rho_plot = np.maximum(rho_all[i], floor)
                ax.plot(x, rho_plot, lw=1)
                ax.set_title(f"T = {temperatures[i]:.4g}", fontsize=9)
                ax.set_xlabel("x")
                ax.set_ylabel(r"$\rho(x)$")
            
            # Turn off unused axes if grid has more slots than plots
            for ax in axes[n_plots:]:
                ax.axis("off")
            
            fig.suptitle(f"Density profiles ({type})")
            fig.tight_layout()
            fig.savefig(os.path.join(log_dir, "rho_profiles_subplots.png"), dpi=200)
            plt.close(fig)



            
surface_tensions = np.full(n_temperatures, np.nan)

for i in range(n_temperatures):
    if type == "AAAA":
        surface_tensions[i] = compute_surface_tension(
            rho_all[i], Da_all[i], dx, floor, B2, K, valence, "AAAA"
        )
    elif type == "ABBB":
        surface_tensions[i] = compute_surface_tension(
            rho_all[i], Da_all[i], dx, floor, B2, K, valence, "ABBB", Db=Db_all[i]
        )
    plt.plot(temperatures, surface_tensions, 'o-')
    plt.xlabel("T")
    plt.ylabel("Surface tension")
    plt.savefig(os.path.join(log_dir, f"surface_tension_vs_T.png"))

/tmp/ipykernel_1173152/3264936517.py:26: RuntimeWarning: invalid value encountered in log
  beta_mu_total = (2.0 * B2 * rho + np.log(rho) + #beta mu_ref


In [ ]:
import matplotlib.pyplot as plt

x = np.arange(gridpoints) * dx

# Select a subset of temperature indices, evenly spaced across the sweep
n_plots = 15  # adjust as needed
plot_indices = np.linspace(0, n_temperatures - 1, n_plots, dtype=int)
plot_indices = np.unique(plot_indices)  # in case n_temperatures < n_plots
n_plots = len(plot_indices)

# Grid layout: roughly square
n_cols = int(np.ceil(np.sqrt(n_plots)))
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3 * n_rows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).flatten()

for ax, i in zip(axes, plot_indices):
    rho_plot = np.maximum(rho_all[i], floor)
    ax.plot(x, rho_plot, lw=1)
    ax.set_title(f"T = {temperatures[i]:.4g}", fontsize=9)
    ax.set_xlabel("x")
    ax.set_ylabel(r"$\rho(x)$")

# Turn off unused axes if grid has more slots than plots
for ax in axes[n_plots:]:
    ax.axis("off")

fig.suptitle(f"Density profiles ({type})")
fig.tight_layout()
fig.savefig(os.path.join(log_dir, "rho_profiles_subplots.png"), dpi=200)
plt.close(fig)